In [ ]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
        .appName("Gold")

        # Delta Lake
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

        # Memory (CRITICAL for 100M+ rows)
        .config("spark.driver.memory", "8g")
        .config("spark.executor.memory", "8g")

        # CPU cores
        .config("spark.driver.cores", "4")

        # Shuffle tuning
        .config("spark.sql.shuffle.partitions", "200")

        # Adaptive query optimization
        .config("spark.sql.adaptive.enabled", "true")

        # Prevent huge driver result crashes
        .config("spark.driver.maxResultSize", "2g")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [ ]:
#importing modules
from pyspark.sql.functions import *
from pyspark.sql.types import *
import os
import logging
import requests

In [ ]:
log_dir = "../logs"
os.makedirs(log_dir, exist_ok=True)

logging.basicConfig(
    filename=os.path.join(log_dir, "03_silver_to_gold.log"),
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

In [ ]:
logging.info(f"Processing Silver Layer")

In [ ]:
df_trip_data = spark.read\
                        .format("delta") \
                        .load(r"..\02_storage_silver\yellow_taxi")

In [ ]:
df_trip_data.count()

### Dimensional Modelling Part 1 - Create Dimensional Tables

From Data Dictionary

In [ ]:
vendor_data = [
    (1, "Creative Mobile Technologies"),
    (2, "Curb Mobility"),
    (6, "Myle Technologies"),
    (7, "Helix")
]

dim_vendor = spark.createDataFrame(vendor_data, ["vendor_id", "vendor_name"])

In [ ]:
rate_data = [
    (1, "Standard"),
    (2, "JFK"),
    (3, "Newark"),
    (4, "Nassau/Westchester"),
    (5, "Negotiated fare"),
    (6, "Group ride"),
    (99, "Unknown")
]

dim_rate = spark.createDataFrame(rate_data, ["rate_code_id", "rate_name"])

In [ ]:
payment_data = [
    (0, "Flex Fare"),
    (1, "Credit Card"),
    (2, "Cash"),
    (3, "No charge"),
    (4, "Dispute"),
    (5, "Unknown"),
    (6, "Voided trip")
]

dim_payment = spark.createDataFrame(payment_data, ["payment_type", "payment_name"])

In [ ]:
store_flag_data = [
    ("Y", "Stored and forwarded"),
    ("N", "Sent directly")
]

dim_store_flag = spark.createDataFrame(
    store_flag_data,
    ["store_and_fwd_flag", "store_flag_desc"]
)

Trip Zone Lookup

In [ ]:
url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
local_path = r"..\00_storage_raw\taxi_zone_lookup\taxi_zone_lookup.csv"
os.makedirs(os.path.dirname(local_path), exist_ok=True)

response = requests.get(url)

with open(local_path, "wb") as f:
    f.write(response.content)
logging.info(f"location dimension pulled from source api")

In [ ]:
dim_location = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(local_path)
)
dim_location.show(5)

In [ ]:
dim_location = dim_location.select(
    col("LocationID").alias("location_id"),
    col("Borough").alias("borough"),
    col("Zone").alias("zone"),
    col("service_zone")
)

Write to dim folder

In [ ]:
def write_dim(df, name):
    (
        df
        .coalesce(1)
        .write
        .format("delta")
        .mode("overwrite")
        .save(fr"..\03_storage_gold\dimensions\{name}")
    )
    logging.info(f"{name} dimension written to Gold layer")

write_dim(dim_vendor, "dim_vendor")
write_dim(dim_payment, "dim_payment_type")
write_dim(dim_rate, "dim_rate_code")
write_dim(dim_store_flag, "dim_store_forward_flag")
write_dim(dim_location, "dim_location")

### Dimensional Modelling Part 2- Build Fact Tables

Tables for dashboard

In [ ]:
fact_trip_yearly_summary = (
    df_trip_data
    .groupBy("trip_year")
    .agg(
        count("*").alias("total_trips"),

        sum("total_amount").alias("total_revenue"),
        avg("total_amount").alias("avg_trip_value"),

        sum("trip_distance").alias("total_distance"),
        avg("trip_distance").alias("avg_distance"),

        avg("fare_per_mile").alias("avg_fare_per_mile"),

        avg("passenger_count").alias("avg_passengers"),

        sum("airport_fee").alias("total_airport_fee"),
        sum("congestion_surcharge").alias("total_congestion_fee"),
        sum("cbd_congestion_fee").alias("total_cbd_fee")
    )
)

fact_trip_yearly_summary.show()

In [ ]:
fact_payment_summary = (
    df_trip_data
    .groupBy("trip_year", "payment_type")
    .agg(
        count("*").alias("total_trips"),

        sum("total_amount").alias("total_revenue"),
        avg("total_amount").alias("avg_trip_value"),

        avg("fare_per_mile").alias("avg_fare_per_mile")
    )
)

fact_payment_summary.show()

In [ ]:
fact_location_summary = (
    df_trip_data
    .groupBy("trip_year", "pu_location_id")
    .agg(
        count("*").alias("total_trips"),

        sum("total_amount").alias("total_revenue"),

        avg("total_amount").alias("avg_trip_value"),
        avg("trip_distance").alias("avg_distance"),

        avg("fare_per_mile").alias("avg_fare_per_mile")
    )
)

fact_location_summary.show()

In [ ]:
df_with_distance_bucket = (
    df_trip_data
    .withColumn(
        "distance_bucket",

        when(col("trip_distance") < 2, "short")
        .when(col("trip_distance") < 10, "medium")
        .otherwise("long")
    )
)

fact_distance_summary = (
    df_with_distance_bucket
    .groupBy("trip_year", "distance_bucket")
    .agg(
        count("*").alias("total_trips"),

        sum("total_amount").alias("total_revenue"),

        avg("total_amount").alias("avg_trip_value"),

        avg("fare_per_mile").alias("avg_fare_per_mile"),

        avg("trip_distance").alias("avg_distance")
    )
)

fact_distance_summary.show()

In [ ]:
fact_trip_time_summary = (
    df_trip_data
    .groupBy(
        "trip_year",
        "pickup_month",
        "pickup_day",
        "pickup_hour"
    )
    .agg(
        count("*").alias("total_trips"),

        sum("total_amount").alias("total_revenue"),
        avg("total_amount").alias("avg_trip_value"),

        avg("trip_distance").alias("avg_distance"),

        avg("fare_per_mile").alias("avg_fare_per_mile"),

        avg("passenger_count").alias("avg_passengers")
    )
)

fact_trip_time_summary.show()

In [ ]:
fact_vendor_summary = (
    df_trip_data
    .groupBy(
        "trip_year",
        "vendor_id"
    )
    .agg(
        count("*").alias("total_trips"),

        sum("total_amount").alias("total_revenue"),

        avg("total_amount").alias("avg_trip_value"),

        avg("trip_distance").alias("avg_distance"),

        avg("fare_per_mile").alias("avg_fare_per_mile"),

        avg("passenger_count").alias("avg_passengers"),

        sum("airport_fee").alias("total_airport_fee"),

        sum("congestion_surcharge").alias("total_congestion_fee"),

        sum("cbd_congestion_fee").alias("total_cbd_fee")
    )
)

fact_vendor_summary.show()

Write to Disk

In [ ]:
def write_fact(df, name):
    (
        df
        .coalesce(1)
        .write
        .format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true")
        .save(fr"..\03_storage_gold\facts\{name}")
    )
    logging.info(f"{name}  written to Gold layer")

In [ ]:
write_fact(fact_trip_yearly_summary, "fact_trip_yearly_summary")
write_fact(fact_payment_summary, "fact_payment_summary")
write_fact(fact_location_summary, "fact_location_summary")
write_fact(fact_distance_summary, "fact_distance_summary")
write_fact(fact_trip_time_summary, "fact_trip_time_summary")
write_fact(fact_vendor_summary, "fact_vendor_summary")